# 🫀 Pipeline de Machine Learning — Doenças Cardíacas

**Objetivo:** Construir um pipeline completo de Machine Learning para **classificação binária** de doenças cardíacas, utilizando o dataset *Heart Disease* da UCI Machine Learning Repository (Cleveland).

---

## Etapa 1 — Escolha e Análise do Dataset

Nesta etapa, carregaremos o dataset, exploraremos sua estrutura e realizaremos uma **Análise Exploratória de Dados (EDA)** detalhada para compreender os dados antes de qualquer modelagem.

### 1.1 Descrição do Problema

As **doenças cardiovasculares (DCVs)** são a principal causa de morte no mundo, responsáveis por aproximadamente **17,9 milhões de óbitos por ano** segundo a OMS. A detecção precoce de indivíduos com alto risco cardíaco é fundamental para intervenções preventivas eficazes.

O dataset **Heart Disease** (Cleveland), disponível no UCI ML Repository, contém registros clínicos de pacientes com atributos como idade, sexo, pressão arterial, colesterol, entre outros. A variável alvo indica a **presença ou ausência de doença cardíaca**, configurando um problema de **classificação binária**.

**Meta:** Treinar um modelo capaz de predizer, com base em atributos clínicos, se um paciente possui ou não doença cardíaca.

### 1.2 Instalação de Dependências e Importações

In [ ]:
# Instalação da biblioteca para acessar datasets da UCI
!pip install -q ucimlrepo

In [ ]:
# --- Importações ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo

# Configurações de visualização
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# Reprodutibilidade
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ Bibliotecas importadas com sucesso.")

### 1.3 Carregamento do Dataset

In [ ]:
# Fetch do dataset Heart Disease (ID 45 no UCI ML Repository)
heart_disease = fetch_ucirepo(id=45)

# Separação entre features e target
X = heart_disease.data.features
y = heart_disease.data.targets

# Montagem do DataFrame completo para análise
df = pd.concat([X, y], axis=1)

# Exibição dos metadados do dataset
print("📋 Informações do Dataset")
print(f"   Nome: {heart_disease.metadata.name}")
print(f"   Área: {heart_disease.metadata.area}")
print(f"   Tarefa associada: {heart_disease.metadata.task}")
print(f"   Número de instâncias: {heart_disease.metadata.num_instances}")
print()

In [ ]:
# Primeiras linhas do dataset
print("🔍 Primeiras 5 amostras do dataset:")
df.head()

### 1.4 Quantidade de Amostras e Atributos

In [ ]:
n_amostras, n_colunas = df.shape
n_features = X.shape[1]
nome_alvo = y.columns[0]

print(f"📊 Dimensões do Dataset")
print(f"   Total de amostras:   {n_amostras}")
print(f"   Total de colunas:    {n_colunas}")
print(f"   Atributos (features): {n_features}")
print(f"   Variável alvo:       '{nome_alvo}'")
print()
print("📝 Tipos dos atributos:")
print(df.dtypes.to_string())

In [ ]:
# Descrição detalhada dos atributos
descricao_atributos = {
    "age": "Idade do paciente (anos)",
    "sex": "Sexo (1 = masculino, 0 = feminino)",
    "cp": "Tipo de dor no peito (0-3)",
    "trestbps": "Pressão arterial em repouso (mm Hg)",
    "chol": "Colesterol sérico (mg/dl)",
    "fbs": "Glicemia em jejum > 120 mg/dl (1 = sim, 0 = não)",
    "restecg": "Resultados do eletrocardiograma em repouso (0-2)",
    "thalach": "Frequência cardíaca máxima atingida",
    "exang": "Angina induzida por exercício (1 = sim, 0 = não)",
    "oldpeak": "Depressão do segmento ST induzida por exercício",
    "slope": "Inclinação do segmento ST no pico do exercício (0-2)",
    "ca": "Número de vasos principais coloridos por fluoroscopia (0-3)",
    "thal": "Talassemia (3 = normal, 6 = defeito fixo, 7 = defeito reversível)",
    "num": "Diagnóstico de doença cardíaca (0 = ausência, 1-4 = presença)"
}

df_desc = pd.DataFrame(
    descricao_atributos.items(),
    columns=["Atributo", "Descrição"]
)
print("📖 Dicionário de Atributos:")
df_desc

### 1.5 Variável Alvo e Distribuição das Classes

A variável alvo original (`num`) possui valores de 0 a 4, onde 0 indica ausência de doença e 1–4 indicam diferentes graus de presença. Para simplificar o problema como **classificação binária**, transformaremos os valores > 0 em 1 (doença presente).

In [ ]:
# Transformação para classificação binária
df["target"] = (df["num"] > 0).astype(int)

# Distribuição das classes
contagem_classes = df["target"].value_counts()
percentual_classes = df["target"].value_counts(normalize=True) * 100

print("🎯 Distribuição da Variável Alvo (target):")
print(f"   Classe 0 (Sem doença):  {contagem_classes[0]} amostras ({percentual_classes[0]:.1f}%)")
print(f"   Classe 1 (Com doença):  {contagem_classes[1]} amostras ({percentual_classes[1]:.1f}%)")
print(f"   Razão (maioria/minoria): {contagem_classes.max() / contagem_classes.min():.2f}")

In [ ]:
# Gráfico de distribuição das classes
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico de barras
cores = ["#2ecc71", "#e74c3c"]
rotulos = ["Sem Doença (0)", "Com Doença (1)"]
bars = axes[0].bar(rotulos, contagem_classes.values, color=cores, edgecolor="white", linewidth=1.5)
axes[0].set_title("Distribuição das Classes", fontweight="bold")
axes[0].set_ylabel("Número de Amostras")
for bar, count in zip(bars, contagem_classes.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
        str(count), ha="center", va="bottom", fontweight="bold", fontsize=13
    )

# Gráfico de pizza
axes[1].pie(
    contagem_classes.values,
    labels=rotulos,
    autopct="%1.1f%%",
    colors=cores,
    startangle=90,
    explode=(0.03, 0.03),
    textprops={"fontsize": 12}
)
axes[1].set_title("Proporção das Classes", fontweight="bold")

plt.tight_layout()
plt.show()

### 1.6 Estatísticas Descritivas

In [ ]:
# Estatísticas descritivas das features numéricas
print("📈 Estatísticas Descritivas:")
df.describe().T.style.background_gradient(cmap="YlOrRd", axis=1)

---

## 1.7 Análise de Problemas no Dataset

Antes de construir qualquer modelo, é essencial identificar problemas que podem comprometer a qualidade do aprendizado.

#### 1.7.1 Valores Ausentes

In [ ]:
# Contagem de valores ausentes por coluna
valores_ausentes = df.isnull().sum()
percentual_ausentes = (df.isnull().sum() / len(df)) * 100

df_ausentes = pd.DataFrame({
    "Valores Ausentes": valores_ausentes,
    "Percentual (%)": percentual_ausentes.round(2)
}).sort_values(by="Valores Ausentes", ascending=False)

print("❓ Valores Ausentes por Atributo:")
print(df_ausentes.to_string())
print(f"\n   Total de células ausentes: {df.isnull().sum().sum()} "
      f"de {df.shape[0] * df.shape[1]} ({df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) * 100:.2f}%)")

In [ ]:
# Visualização dos valores ausentes
colunas_com_nulos = df_ausentes[df_ausentes["Valores Ausentes"] > 0]

if len(colunas_com_nulos) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(
        colunas_com_nulos.index,
        colunas_com_nulos["Valores Ausentes"],
        color="#e67e22",
        edgecolor="white"
    )
    ax.set_xlabel("Quantidade de Valores Ausentes")
    ax.set_title("Atributos com Valores Ausentes", fontweight="bold")
    for bar, pct in zip(bars, colunas_com_nulos["Percentual (%)"]):
        ax.text(
            bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{pct:.1f}%", va="center", fontweight="bold"
        )
    plt.tight_layout()
    plt.show()
else:
    print("✅ Nenhum valor ausente encontrado no dataset!")

#### 1.7.2 Detecção de Desbalanceamento

In [ ]:
# Análise de desbalanceamento
razao = contagem_classes.max() / contagem_classes.min()

print("⚖️  Análise de Desbalanceamento:")
print(f"   Classe majoritária: {contagem_classes.idxmax()} ({contagem_classes.max()} amostras)")
print(f"   Classe minoritária: {contagem_classes.idxmin()} ({contagem_classes.min()} amostras)")
print(f"   Razão de desbalanceamento: {razao:.2f}:1")
print()

if razao < 1.5:
    print("   ✅ O dataset está BALANCEADO (razão < 1.5).")
    print("   → Não será necessário aplicar técnicas de reamostragem.")
elif razao < 3.0:
    print("   ⚠️  O dataset apresenta LEVE DESBALANCEAMENTO (1.5 ≤ razão < 3.0).")
    print("   → Considerar ajuste de pesos nas classes ou métricas como F1-Score.")
else:
    print("   🚨 O dataset está DESBALANCEADO (razão ≥ 3.0).")
    print("   → Recomenda-se aplicar SMOTE, undersampling ou ajuste de class_weight.")

#### 1.7.3 Identificação de Outliers (Boxplots)

In [ ]:
# Seleção de colunas numéricas contínuas para análise de outliers
colunas_continuas = ["age", "trestbps", "chol", "thalach", "oldpeak"]

fig, axes = plt.subplots(1, len(colunas_continuas), figsize=(18, 6))
fig.suptitle("Boxplots — Identificação de Outliers", fontsize=16, fontweight="bold", y=1.02)

for i, col in enumerate(colunas_continuas):
    sns.boxplot(
        data=df, y=col, ax=axes[i],
        color="#3498db", flierprops={"marker": "o", "markerfacecolor": "red", "markersize": 6}
    )
    axes[i].set_title(col, fontweight="bold")
    axes[i].set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
# Quantificação de outliers pelo método IQR
print("🔎 Contagem de Outliers (Método IQR):")
print("   Atributo       | Outliers | % do Total")
print("   " + "-" * 42)

outliers_resumo = {}
for col in colunas_continuas:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < limite_inf) | (df[col] > limite_sup)).sum()
    pct = (n_outliers / len(df)) * 100
    outliers_resumo[col] = n_outliers
    print(f"   {col:<16} | {n_outliers:>8} | {pct:>6.1f}%")

print(f"\n   Total de outliers detectados: {sum(outliers_resumo.values())}")

In [ ]:
# Boxplots separados por classe (target)
fig, axes = plt.subplots(1, len(colunas_continuas), figsize=(18, 6))
fig.suptitle("Boxplots por Classe — Comparação entre Doentes e Saudáveis",
             fontsize=16, fontweight="bold", y=1.02)

for i, col in enumerate(colunas_continuas):
    sns.boxplot(
        data=df, x="target", y=col, ax=axes[i],
        palette={0: "#2ecc71", 1: "#e74c3c"},
        flierprops={"marker": "o", "markerfacecolor": "orange", "markersize": 5}
    )
    axes[i].set_title(col, fontweight="bold")
    axes[i].set_xlabel("")
    axes[i].set_xticklabels(["Saudável", "Doente"])
    axes[i].set_ylabel("")

plt.tight_layout()
plt.show()

#### 1.7.4 Heatmap de Correlação

In [ ]:
# Heatmap de correlação entre atributos numéricos
# Selecionar apenas colunas numéricas (excluindo a coluna 'num' original, mantendo 'target')
colunas_numericas = df.select_dtypes(include=[np.number]).columns.tolist()
if "num" in colunas_numericas:
    colunas_numericas.remove("num")

corr_matrix = df[colunas_numericas].corr()

# Máscara para triângulo superior (evitar redundância)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8, "label": "Coeficiente de Correlação"},
    ax=ax
)
ax.set_title("Heatmap de Correlação entre Atributos", fontsize=16, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Top correlações com a variável alvo
corr_com_target = corr_matrix["target"].drop("target").abs().sort_values(ascending=False)

print("🏆 Correlação dos Atributos com a Variável Alvo (|r|):")
print()
for attr, valor in corr_com_target.items():
    barra = "█" * int(valor * 40)
    sinal = "+" if corr_matrix.loc[attr, "target"] > 0 else "-"
    print(f"   {attr:<12} {sinal} {valor:.3f}  {barra}")

print("\n   Atributos com maior poder preditivo (|r| > 0.3):")
fortes = corr_com_target[corr_com_target > 0.3]
for attr in fortes.index:
    print(f"   → {attr}")

---

## 📝 Resumo dos Achados da Análise Exploratória

Após a análise exploratória do dataset **Heart Disease (Cleveland — UCI)**, os principais achados são:

### Dataset
| Item | Valor |
|------|-------|
| **Origem** | UCI ML Repository (Cleveland) |
| **Amostras** | ~303 registros de pacientes |
| **Atributos** | 13 features clínicas + 1 variável alvo |
| **Tipo de problema** | Classificação binária |
| **Variável alvo** | `target` (0 = sem doença, 1 = com doença) |

### Distribuição das Classes
- O dataset apresenta uma distribuição **relativamente balanceada** entre as classes.
- A razão entre a classe majoritária e a minoritária é próxima de 1:1, o que é favorável para o treinamento de modelos sem necessidade imediata de técnicas de reamostragem.

### Valores Ausentes
- Foram identificados valores ausentes em algumas colunas (notavelmente `ca` e `thal`).
- Na próxima etapa, esses valores serão tratados com estratégias de imputação adequadas.

### Outliers
- Variáveis como `chol` (colesterol), `trestbps` (pressão arterial) e `oldpeak` apresentam outliers identificados pelo método IQR.
- Esses outliers podem representar condições clínicas reais e devem ser tratados com cautela (não removidos cegamente).

### Correlações Relevantes
- Os atributos com maior correlação com a variável alvo incluem variáveis relacionadas à dor no peito (`cp`), frequência cardíaca máxima (`thalach`), depressão do ST (`oldpeak`) e angina induzida por exercício (`exang`).
- Não foram observadas correlações extremas entre features (multicolinearidade severa), o que é positivo.

### Próximos Passos
- **Etapa 2:** Pré-processamento dos dados (tratamento de valores ausentes, encoding de variáveis categóricas, normalização/padronização e divisão treino/teste).
- **Etapa 3:** Treinamento e comparação de modelos de classificação.

---

# Etapa 2 — Pré-processamento dos Dados

Com base nos problemas identificados na Etapa 1, aplicaremos um pipeline de pré-processamento estruturado e modular. Cada transformação será implementada como **função reutilizável**, garantindo rastreabilidade e reprodutibilidade.

## 2.1 Justificativa Detalhada das Técnicas de Pré-processamento

Cada decisão de pré-processamento impacta diretamente o comportamento matemático dos algoritmos de aprendizado, especialmente **Redes Neurais Artificiais (RNAs)**. Abaixo, justificamos cada técnica escolhida.

---

### 2.1.1 Tratamento de Valores Ausentes — Imputação pela Mediana

**Técnica escolhida:** `SimpleImputer(strategy="median")`

**Por que a mediana e não a média?**

Na Etapa 1, identificamos outliers significativos em variáveis como `chol`, `trestbps` e `oldpeak`. A **média aritmética** é altamente sensível a valores extremos:

$$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$$

Um único outlier pode deslocar a média substancialmente. Já a **mediana** — o valor central da distribuição ordenada — é uma medida **robusta** de tendência central, pois não é afetada por valores extremos. Isso garante que os dados imputados reflitam melhor a distribuição real dos pacientes.

**Por que não excluir as linhas?**

O dataset possui apenas ~303 amostras. Excluir registros com valores ausentes (especialmente em `ca` e `thal`, que têm ~1-2% de ausências) reduziria o volume de dados disponível para treinamento, prejudicando a capacidade de generalização do modelo.

**Impacto para Redes Neurais:**
- RNAs não aceitam valores `NaN` como entrada — a imputação é **obrigatória**.
- Valores imputados pela mediana mantêm a distribuição mais estável, evitando que os pesos da rede sejam atualizados com sinais enviesados por outliers durante o backpropagation.

---

### 2.1.2 Codificação de Variáveis Categóricas — Análise de Necessidade

**Variáveis categóricas nominais no dataset:**
- `cp` (tipo de dor no peito): 4 categorias (0-3)
- `restecg` (resultado do ECG): 3 categorias (0-2)
- `slope` (inclinação do ST): 3 categorias (0-2)
- `thal` (talassemia): 3 categorias (3, 6, 7)

**Variáveis binárias (já codificadas):**
- `sex`, `fbs`, `exang`: já são 0/1 — não necessitam transformação.

**Técnica escolhida:** `pd.get_dummies()` (One-Hot Encoding) para variáveis **nominais multi-classe** (`cp`, `thal`, `restecg`, `slope`).

**Por que One-Hot Encoding e não Label Encoding?**

O Label Encoding atribui valores numéricos sequenciais (0, 1, 2, 3...), o que **implica uma ordem** entre as categorias. Para variáveis como `cp` (tipo de dor no peito), os valores 0-3 representam tipos diferentes, **sem relação ordinal intrínseca**. Ao usar Label Encoding, a rede neural interpretaria que tipo 3 é "três vezes maior" que tipo 1, introduzindo uma relação falsa.

O One-Hot Encoding cria vetores binários ortogonais:

$$\text{cp}=2 \rightarrow [0, 0, 1, 0]$$

Isso garante que cada categoria esteja **equidistante** no espaço de features, sem viés ordinal.

**Impacto para Redes Neurais:**
- Cada neurônio da camada de entrada recebe um sinal binário (0 ou 1) por categoria, permitindo que a rede aprenda **pesos independentes** para cada tipo.
- Evita que o gradiente descendente otimize pesos baseado em magnitudes numéricas artificiais.
- Trade-off: aumenta a dimensionalidade, mas com apenas 4 variáveis categóricas de poucos níveis, o impacto é mínimo.

**Nota:** Utilizaremos `drop_first=True` para evitar **multicolinearidade perfeita** (a "armadilha das variáveis dummy"), que pode causar instabilidade numérica na matriz de pesos.

---

### 2.1.3 Padronização das Features — RobustScaler

**Técnica escolhida:** `RobustScaler` do scikit-learn.

A padronização transforma cada feature para uma escala comum. O `RobustScaler` utiliza a **mediana** e o **intervalo interquartil (IQR)** em vez da média e desvio padrão:

$$x_{\text{scaled}} = \frac{x - \text{mediana}(x)}{\text{IQR}(x)} = \frac{x - Q_2}{Q_3 - Q_1}$$

**Por que RobustScaler e não StandardScaler?**

O `StandardScaler` utiliza a transformação Z-score:

$$z = \frac{x - \mu}{\sigma}$$

Como tanto $\mu$ (média) quanto $\sigma$ (desvio padrão) são sensíveis a outliers, o StandardScaler pode produzir escalas distorcidas. O `RobustScaler`, baseado em estatísticas de posição robustas (mediana e quartis), é **imune à influência de outliers**, mantendo a escala coerente mesmo na presença de valores extremos detectados na Etapa 1.

**Impacto para Redes Neurais:**

1. **Convergência do gradiente descendente:** Sem padronização, features com escalas muito diferentes (ex: `chol` em [100, 600] vs `oldpeak` em [0, 6]) fazem com que a superfície da função de custo (loss) seja alongada ("elíptica"), forçando o otimizador a oscilar ao longo dos eixos com maior escala. Com features padronizadas, a superfície se torna mais "esférica", permitindo convergência **mais rápida e estável**.

2. **Inicialização dos pesos:** Técnicas modernas de inicialização (Xavier, He) assumem que os dados de entrada possuem média prox. de 0 e variância prox. de 1. Dados não padronizados violam essa premissa, resultando em **saturação de neurônios** (especialmente com funções de ativação sigmoide/tanh) ou **gradientes explosivos**.

3. **Regularização justa:** Técnicas de regularização (L1/L2) penalizam pesos proporcionalmente à sua magnitude. Se features têm escalas diferentes, a penalização será desproporcional — features com escala maior terão pesos naturalmente menores e serão menos penalizadas, distorcendo o aprendizado.

4. **Nota sobre colunas One-Hot:** As colunas geradas pelo One-Hot Encoding (0/1) **não serão padronizadas**, pois já estão em escala binária uniforme. Padronizá-las seria contraproducente, pois distorceria sua interpretação semântica.

---

## 2.2 Importações Adicionais para Pré-processamento

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split

print("✅ Importações de pré-processamento carregadas.")

## 2.3 Preparação do DataFrame de Trabalho

Partimos do DataFrame `df` construído na Etapa 1. Removemos a coluna `num` original (mantendo apenas `target` como variável alvo binária).

In [ ]:
# Cópia do DataFrame original para preservar os dados brutos
df_proc = df.drop(columns=["num"]).copy()

print(f"📋 DataFrame de trabalho: {df_proc.shape[0]} amostras x {df_proc.shape[1]} colunas")
print(f"   Colunas: {list(df_proc.columns)}")
print()
print("🔍 Estado atual dos valores ausentes:")
nulos = df_proc.isnull().sum()
nulos_existentes = nulos[nulos > 0]
if len(nulos_existentes) > 0:
    print(nulos_existentes.to_string())
else:
    print("   Nenhum valor ausente.")
print()
df_proc.head()

---

## 2.4 Funções Modulares de Pré-processamento

Todas as transformações são encapsuladas em funções reutilizáveis, seguindo boas práticas de engenharia de ML.

In [ ]:
def tratar_valores_ausentes(dataframe, estrategia="median"):
    """
    Imputa valores ausentes em colunas numericas usando a estrategia especificada.

    Parametros:
        dataframe (pd.DataFrame): DataFrame de entrada.
        estrategia (str): Estrategia de imputacao ('median', 'mean', 'most_frequent').

    Retorna:
        pd.DataFrame: DataFrame com valores ausentes tratados.
        SimpleImputer: O objeto imputer ajustado (para aplicar nos dados de teste).
        list: Lista de colunas que receberam imputacao.
    """
    df_out = dataframe.copy()
    colunas_com_nulos = df_out.columns[df_out.isnull().any()].tolist()

    if not colunas_com_nulos:
        print("   ✅ Nenhum valor ausente encontrado. Nenhuma imputacao necessaria.")
        return df_out, None, []

    imputer = SimpleImputer(strategy=estrategia)
    df_out[colunas_com_nulos] = imputer.fit_transform(df_out[colunas_com_nulos])

    print(f"   ✅ Imputação aplicada (estratégia: {estrategia}) nas colunas:")
    for col in colunas_com_nulos:
        n_imputados = dataframe[col].isnull().sum()
        mediana_usada = np.nanmedian(dataframe[col])
        print(f"      → {col}: {n_imputados} valores imputados (mediana = {mediana_usada:.2f})")

    return df_out, imputer, colunas_com_nulos


def codificar_categoricas(dataframe, colunas_categoricas, drop_first=True):
    """
    Aplica One-Hot Encoding nas colunas categoricas especificadas.

    Parametros:
        dataframe (pd.DataFrame): DataFrame de entrada.
        colunas_categoricas (list): Nomes das colunas categoricas.
        drop_first (bool): Se True, remove a primeira categoria (evita multicolinearidade).

    Retorna:
        pd.DataFrame: DataFrame com variaveis dummy.
    """
    df_out = dataframe.copy()
    n_antes = df_out.shape[1]

    df_out = pd.get_dummies(
        df_out,
        columns=colunas_categoricas,
        drop_first=drop_first,
        dtype=int
    )

    n_depois = df_out.shape[1]
    print(f"   ✅ One-Hot Encoding aplicado.")
    print(f"      Colunas antes: {n_antes} → Colunas depois: {n_depois} (+{n_depois - n_antes})")
    print(f"      Variáveis codificadas: {colunas_categoricas}")
    print(f"      drop_first={drop_first} → evita armadilha das variáveis dummy")

    return df_out


def padronizar_features(dataframe, colunas_numericas, colunas_excluir=None):
    """
    Aplica RobustScaler nas colunas numericas continuas, excluindo binarias e target.

    Parametros:
        dataframe (pd.DataFrame): DataFrame de entrada.
        colunas_numericas (list): Nomes das colunas numericas a padronizar.
        colunas_excluir (list): Colunas que nao devem ser padronizadas.

    Retorna:
        pd.DataFrame: DataFrame com features padronizadas.
        RobustScaler: O objeto scaler ajustado.
    """
    df_out = dataframe.copy()
    if colunas_excluir:
        colunas_a_escalar = [c for c in colunas_numericas if c not in colunas_excluir]
    else:
        colunas_a_escalar = colunas_numericas

    scaler = RobustScaler()
    df_out[colunas_a_escalar] = scaler.fit_transform(df_out[colunas_a_escalar])

    print(f"   ✅ RobustScaler aplicado em {len(colunas_a_escalar)} colunas:")
    print(f"      {colunas_a_escalar}")

    return df_out, scaler


print("✅ Funções de pré-processamento definidas.")

---

## 2.5 Aplicação do Pipeline de Pré-processamento

### Passo 1 — Tratamento de Valores Ausentes

In [ ]:
print("🔧 PASSO 1: Tratamento de Valores Ausentes")
print("=" * 50)
print()

# Estado antes da imputação
print("   📊 Antes da imputação:")
print(f"      Valores ausentes totais: {df_proc.isnull().sum().sum()}")
print()

# Aplicação da imputação pela mediana
df_proc, imputer_fitted, colunas_imputadas = tratar_valores_ausentes(df_proc, estrategia="median")

print()
print("   📊 Após a imputação:")
print(f"      Valores ausentes totais: {df_proc.isnull().sum().sum()}")

### Passo 2 — Codificação de Variáveis Categóricas

In [ ]:
print("🔧 PASSO 2: Codificação de Variáveis Categóricas (One-Hot Encoding)")
print("=" * 60)
print()

# Definição das variáveis categóricas nominais (multi-classe)
# Variáveis binárias (sex, fbs, exang) já estão codificadas como 0/1
COLUNAS_CATEGORICAS = ["cp", "restecg", "slope", "thal"]

print(f"   📋 Variáveis categóricas identificadas: {COLUNAS_CATEGORICAS}")
print('   📋 Variáveis binárias (já codificadas): ["sex", "fbs", "exang"]')
print()

# Exibir valores únicos antes da codificação
for col in COLUNAS_CATEGORICAS:
    print(f"      {col}: {sorted(df_proc[col].dropna().unique())}")
print()

# Aplicação do One-Hot Encoding
df_proc = codificar_categoricas(df_proc, COLUNAS_CATEGORICAS, drop_first=True)

print()
print(f"   📋 Novas colunas do DataFrame: {list(df_proc.columns)}")

### Passo 3 — Padronização das Features Numéricas

In [ ]:
print("🔧 PASSO 3: Padronização das Features Numéricas (RobustScaler)")
print("=" * 60)
print()

# Colunas numéricas contínuas que devem ser padronizadas
COLUNAS_NUMERICAS_CONTINUAS = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]

# Colunas que NAO devem ser padronizadas (binárias, one-hot, target)
colunas_binarias_e_onehot = [
    c for c in df_proc.columns
    if c not in COLUNAS_NUMERICAS_CONTINUAS + ["target"]
]
print(f"   📋 Colunas a padronizar: {COLUNAS_NUMERICAS_CONTINUAS}")
print(f"   📋 Colunas NAO padronizadas (binárias/one-hot): {colunas_binarias_e_onehot}")
print()

# Estatísticas ANTES da padronização
print("   📊 Estatísticas ANTES da padronização:")
stats_antes = df_proc[COLUNAS_NUMERICAS_CONTINUAS].describe().loc[["mean", "std", "min", "max"]]
print(stats_antes.round(2).to_string())
print()

# Aplicação do RobustScaler
df_proc, scaler_fitted = padronizar_features(
    df_proc,
    colunas_numericas=COLUNAS_NUMERICAS_CONTINUAS
)

print()
print("   📊 Estatísticas APÓS a padronização:")
stats_depois = df_proc[COLUNAS_NUMERICAS_CONTINUAS].describe().loc[["mean", "std", "min", "max"]]
print(stats_depois.round(2).to_string())

---

## 2.6 Visualização Comparativa: Antes vs. Após a Padronização

In [ ]:
# Distribuições antes e depois da padronização
# Recriamos os dados originais (sem padronização) para comparação
df_antes = df.drop(columns=["num"]).copy()
if colunas_imputadas:
    df_antes[colunas_imputadas] = imputer_fitted.transform(
        df_antes[colunas_imputadas]
    )

fig, axes = plt.subplots(
    2, len(COLUNAS_NUMERICAS_CONTINUAS), figsize=(22, 8)
)
fig.suptitle(
    "Distribuições: Antes vs. Após Padronização (RobustScaler)",
    fontsize=16, fontweight="bold", y=1.02
)

for i, col in enumerate(COLUNAS_NUMERICAS_CONTINUAS):
    # Antes
    axes[0, i].hist(
        df_antes[col].dropna(), bins=20,
        color="#3498db", alpha=0.7, edgecolor="white"
    )
    axes[0, i].set_title(f"{col}\n(original)", fontsize=11, fontweight="bold")
    axes[0, i].axvline(
        df_antes[col].median(), color="red",
        linestyle="--", linewidth=1.5, label="Mediana"
    )
    if i == 0:
        axes[0, i].set_ylabel("ANTES", fontsize=12, fontweight="bold")

    # Depois
    axes[1, i].hist(
        df_proc[col].dropna(), bins=20,
        color="#2ecc71", alpha=0.7, edgecolor="white"
    )
    axes[1, i].set_title(
        f"{col}\n(padronizado)", fontsize=11, fontweight="bold"
    )
    axes[1, i].axvline(
        df_proc[col].median(), color="red",
        linestyle="--", linewidth=1.5, label="Mediana"
    )
    if i == 0:
        axes[1, i].set_ylabel("DEPOIS", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

## 2.7 Divisão Treino/Teste e Estado Final

In [ ]:
# Separação entre features (X) e target (y)
X_final = df_proc.drop(columns=["target"])
y_final = df_proc["target"]

# Divisão treino/teste estratificada (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_final,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_final
)

print("📦 Divisão Treino/Teste (80/20 — estratificada):")
print(f"   X_train: {X_train.shape}")
print(f"   X_test:  {X_test.shape}")
print(f"   y_train: {y_train.shape} — distribuição: {dict(y_train.value_counts())}")
print(f"   y_test:  {y_test.shape} — distribuição: {dict(y_test.value_counts())}")
print()
print(f"   Features finais ({X_final.shape[1]}): {list(X_final.columns)}")

In [ ]:
# Verificação final do DataFrame processado
print("🔍 Verificação Final do Dataset Pré-processado:")
print("=" * 50)
print(f"   Valores ausentes: {df_proc.isnull().sum().sum()}")
print(f"   Linhas duplicadas: {df_proc.duplicated().sum()}")
print(f"   Dimensão final: {df_proc.shape}")
print(f"   Tipos de dados:")
print(f"   {df_proc.dtypes.value_counts().to_string()}")
print()
print("📋 Amostra do dataset processado:")
df_proc.head()

---

## 📝 Resumo da Etapa 2 — Pré-processamento

### Transformações Aplicadas

| Passo | Técnica | Justificativa | Colunas Afetadas |
|-------|---------|---------------|------------------|
| **1** | Imputação pela Mediana (`SimpleImputer`) | Robusta a outliers; preserva distribuição central sem perder amostras | `ca`, `thal` |
| **2** | One-Hot Encoding (`pd.get_dummies`) | Evita relações ordinais falsas entre categorias nominais | `cp`, `restecg`, `slope`, `thal` |
| **3** | Padronização (`RobustScaler`) | Escala baseada em mediana/IQR — resistente a outliers; essencial para convergência de RNAs | `age`, `trestbps`, `chol`, `thalach`, `oldpeak`, `ca` |
| **4** | Divisão estratificada (80/20) | Mantém proporção das classes em treino e teste | Todas |

### Impacto Matemático para Redes Neurais

- **Imputação:** Elimina `NaN` (obrigatório para RNAs) sem enviesar distribuição.
- **One-Hot Encoding:** Garante equidistância entre categorias no espaço de entrada — pesos independentes por categoria.
- **RobustScaler:** Normaliza a superfície da função de custo, acelerando convergência do gradiente descendente e prevenindo saturação de neurônios.
- **Divisão estratificada:** Garante que treino e teste representem fielmente a distribuição original das classes.

### Próximos Passos

- **Etapa 3:** Treinamento e comparação de modelos de classificação (Regressão Logística, SVM, Random Forest, Rede Neural).
- Os objetos `imputer_fitted`, `scaler_fitted` e a lista de features serão reutilizados na Etapa 3 para garantir consistência entre treino e teste.

---

# Etapa 3 — Seleção de Features

Nesta etapa, aplicaremos técnicas de **seleção de atributos** para identificar as features mais relevantes para a classificação de doenças cardíacas. Compararemos o desempenho de um modelo baseline com todas as features versus apenas as selecionadas, e utilizaremos **SHAP** para interpretabilidade.

## 3.1 Importações Adicionais

In [ ]:
!pip install -q shap

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
import shap
import warnings
warnings.filterwarnings("ignore")

print("✅ Importações da Etapa 3 carregadas com sucesso.")

## 3.2 Método Embedded — Importância de Features (Random Forest)

O Random Forest calcula a **importância de cada feature** com base na redução média de impureza (Gini) nas árvores de decisão. Features que geram splits mais informativos recebem maior importância.

Este é um método **embedded**, pois a seleção ocorre como parte do próprio processo de treinamento do modelo.

In [ ]:
# Treinamento de um Random Forest para extrair importancia das features
rf_importancia = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_importancia.fit(X_train, y_train)

# Extrair importancias
importancias = pd.Series(
    rf_importancia.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print("🏆 Ranking de Importância das Features (Random Forest - Gini):")
print("=" * 55)
for rank, (feat, imp) in enumerate(importancias.items(), 1):
    barra = "█" * int(imp * 100)
    print(f"   {rank:2d}. {feat:<20} {imp:.4f}  {barra}")

In [ ]:
# Visualizacao da importancia das features
fig, ax = plt.subplots(figsize=(12, 7))

cores_barra = ["#e74c3c" if v >= importancias.quantile(0.75) else
               "#f39c12" if v >= importancias.quantile(0.50) else
               "#3498db" for v in importancias.values]

bars = ax.barh(
    importancias.index[::-1],
    importancias.values[::-1],
    color=cores_barra[::-1],
    edgecolor="white",
    linewidth=0.5
)
ax.set_xlabel("Importância (Redução Média de Impureza - Gini)")
ax.set_title("Importância das Features — Random Forest", fontweight="bold", fontsize=14)

# Legenda de cores
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#e74c3c", label="Top 25% (alta importância)"),
    Patch(facecolor="#f39c12", label="25-50% (média importância)"),
    Patch(facecolor="#3498db", label="< 50% (baixa importância)")
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=10)

plt.tight_layout()
plt.show()

## 3.3 Método Wrapper — Recursive Feature Elimination (RFE)

O **RFE (Recursive Feature Elimination)** é um método wrapper que treina o modelo repetidamente, removendo a feature menos importante a cada iteração, até atingir o número desejado de features.

Utilizaremos o Random Forest como estimador base do RFE para selecionar as **top features**.

In [ ]:
# Definir numero de features a selecionar
# Regra: selecionar features que acumulem >= 85% da importancia total
importancia_cumulativa = importancias.cumsum()
n_features_85 = (importancia_cumulativa <= 0.85).sum() + 1
n_features_selecionadas = max(n_features_85, 5)  # minimo de 5 features

print(f"📊 Importância cumulativa das features:")
for feat, (imp, cum) in zip(importancias.index,
                             zip(importancias.values, importancia_cumulativa.values)):
    marker = " ← corte 85%" if abs(cum - importancia_cumulativa.iloc[n_features_85 - 1]) < 1e-10 else ""
    print(f"   {feat:<20} {imp:.4f}   cumulativo: {cum:.4f}{marker}")

print(f"\n   Features necessárias para 85% da importância: {n_features_85}")
print(f"   Features a selecionar via RFE: {n_features_selecionadas}")

In [ ]:
# Aplicacao do RFE
estimador_rfe = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rfe = RFE(
    estimator=estimador_rfe,
    n_features_to_select=n_features_selecionadas,
    step=1
)
rfe.fit(X_train, y_train)

# Resultados do RFE
rfe_ranking = pd.DataFrame({
    "Feature": X_train.columns,
    "Selecionada": rfe.support_,
    "Ranking RFE": rfe.ranking_
}).sort_values("Ranking RFE")

features_selecionadas = X_train.columns[rfe.support_].tolist()
features_removidas = X_train.columns[~rfe.support_].tolist()

print("🔍 Resultado do RFE (Recursive Feature Elimination):")
print("=" * 55)
print(f"   Atributos iniciais:     {X_train.shape[1]}")
print(f"   Atributos selecionados: {len(features_selecionadas)}")
print(f"   Atributos removidos:    {len(features_removidas)}")
print()
print("   ✅ Features SELECIONADAS:")
for f in features_selecionadas:
    imp_val = importancias.get(f, 0)
    print(f"      → {f:<20} (importância RF: {imp_val:.4f})")
print()
print("   ❌ Features REMOVIDAS:")
for f in features_removidas:
    imp_val = importancias.get(f, 0)
    print(f"      → {f:<20} (importância RF: {imp_val:.4f})")

In [ ]:
# Tabela completa do ranking RFE
print("\n📋 Ranking completo do RFE:")
rfe_ranking.style.applymap(
    lambda x: "background-color: #d4edda" if x == True
    else ("background-color: #f8d7da" if x == False else ""),
    subset=["Selecionada"]
)

---

## 3.4 Comparação: Todas as Features vs. Features Selecionadas

Treinaremos um **Random Forest baseline** em dois cenários:
1. Com **todas** as features originais
2. Apenas com as **features selecionadas** pelo RFE

Isso nos permite avaliar se a seleção de features melhora o desempenho ou reduz overfitting.

In [ ]:
def treinar_e_avaliar(X_tr, X_te, y_tr, y_te, nome_cenario):
    """
    Treina um Random Forest e retorna as metricas de avaliacao.

    Parametros:
        X_tr, X_te: Features de treino e teste.
        y_tr, y_te: Target de treino e teste.
        nome_cenario (str): Nome descritivo do cenario.

    Retorna:
        dict: Metricas de avaliacao.
        model: Modelo treinado.
    """
    modelo = RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    modelo.fit(X_tr, y_tr)

    # Predicoes
    y_pred = modelo.predict(X_te)
    y_proba = modelo.predict_proba(X_te)[:, 1]

    # Metricas
    metricas = {
        "Cenario": nome_cenario,
        "N. Features": X_tr.shape[1],
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred),
        "Recall": recall_score(y_te, y_pred),
        "F1-Score": f1_score(y_te, y_pred),
        "ROC-AUC": roc_auc_score(y_te, y_proba),
        "Acc. Treino": accuracy_score(y_tr, modelo.predict(X_tr))
    }

    return metricas, modelo


print("✅ Função de treino e avaliação definida.")

In [ ]:
# Cenario 1: Todas as features
metricas_todas, modelo_todas = treinar_e_avaliar(
    X_train, X_test, y_train, y_test,
    nome_cenario="Todas as Features"
)

# Cenario 2: Features selecionadas pelo RFE
X_train_sel = X_train[features_selecionadas]
X_test_sel = X_test[features_selecionadas]

metricas_sel, modelo_sel = treinar_e_avaliar(
    X_train_sel, X_test_sel, y_train, y_test,
    nome_cenario="Features Selecionadas (RFE)"
)

# Tabela comparativa
df_comparacao = pd.DataFrame([metricas_todas, metricas_sel]).set_index("Cenario")

print("📊 Comparação de Desempenho:")
print("=" * 70)
df_comparacao

In [ ]:
# Visualizacao comparativa das metricas
metricas_plot = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(metricas_plot))
largura = 0.35

bars1 = ax.bar(
    x - largura/2,
    [metricas_todas[m] for m in metricas_plot],
    largura, label=f"Todas ({metricas_todas['N. Features']} features)",
    color="#3498db", edgecolor="white", linewidth=1.2
)
bars2 = ax.bar(
    x + largura/2,
    [metricas_sel[m] for m in metricas_plot],
    largura, label=f"Selecionadas ({metricas_sel['N. Features']} features)",
    color="#2ecc71", edgecolor="white", linewidth=1.2
)

# Valores sobre as barras
for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2, h + 0.005,
            f"{h:.3f}", ha="center", va="bottom",
            fontweight="bold", fontsize=10
        )

ax.set_xticks(x)
ax.set_xticklabels(metricas_plot, fontsize=12)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Score")
ax.set_title("Comparação: Todas as Features vs. Features Selecionadas (RFE)",
             fontweight="bold", fontsize=14)
ax.legend(fontsize=11, loc="upper left")
ax.axhline(y=1.0, color="gray", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Analise de overfitting (gap treino vs teste)
gap_todas = metricas_todas["Acc. Treino"] - metricas_todas["Accuracy"]
gap_sel = metricas_sel["Acc. Treino"] - metricas_sel["Accuracy"]

print("🔍 Análise de Overfitting (Gap Treino - Teste):")
print("=" * 55)
print(f"   Todas as features:")
print(f"      Acc. Treino: {metricas_todas['Acc. Treino']:.4f}")
print(f"      Acc. Teste:  {metricas_todas['Accuracy']:.4f}")
print(f"      Gap:         {gap_todas:.4f} {'(⚠️ possível overfitting)' if gap_todas > 0.05 else '(✅ aceitável)'}")
print()
print(f"   Features selecionadas:")
print(f"      Acc. Treino: {metricas_sel['Acc. Treino']:.4f}")
print(f"      Acc. Teste:  {metricas_sel['Accuracy']:.4f}")
print(f"      Gap:         {gap_sel:.4f} {'(⚠️ possível overfitting)' if gap_sel > 0.05 else '(✅ aceitável)'}")
print()
if gap_sel < gap_todas:
    print(f"   ✅ A seleção de features REDUZIU o gap de overfitting em {gap_todas - gap_sel:.4f}")
else:
    print(f"   ℹ️ O gap de overfitting se manteve similar entre os dois cenários.")

In [ ]:
# Matrizes de confusao lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, modelo_c, X_te, titulo in [
    (axes[0], modelo_todas, X_test, f"Todas as Features ({metricas_todas['N. Features']})"),
    (axes[1], modelo_sel, X_test_sel, f"Features Selecionadas ({metricas_sel['N. Features']})")
]:
    y_pred_c = modelo_c.predict(X_te)
    cm = confusion_matrix(y_test, y_pred_c)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Saudável", "Doente"])
    disp.plot(ax=ax, cmap="Blues", values_format="d")
    ax.set_title(titulo, fontweight="bold", fontsize=12)

plt.suptitle("Matrizes de Confusão — Comparação", fontweight="bold", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## 3.5 Interpretabilidade com SHAP

**SHAP (SHapley Additive exPlanations)** é um framework baseado na teoria dos jogos (valores de Shapley) que atribui a cada feature uma contribuição para a predição individual. Isso permite:

- **Explicação global:** Quais features são mais importantes no geral?
- **Explicação local:** Por que o modelo fez *esta* predição específica?

Utilizaremos o `TreeExplainer` otimizado para modelos baseados em árvores.

In [ ]:
# Calcular SHAP values usando o modelo com features selecionadas
explainer = shap.TreeExplainer(modelo_sel)
shap_values = explainer.shap_values(X_test_sel)

# Para classificacao binaria, shap_values retorna uma lista [classe_0, classe_1]
# Usamos a classe 1 (doenca presente)
if isinstance(shap_values, list):
    shap_values_classe1 = shap_values[1]
else:
    shap_values_classe1 = shap_values

print(f"✅ SHAP values calculados para {X_test_sel.shape[0]} amostras de teste.")
print(f"   Shape: {shap_values_classe1.shape}")

### 3.5.1 SHAP Summary Plot (Explicação Global)

In [ ]:
# SHAP Summary Plot - Beeswarm
print("📊 SHAP Summary Plot — Impacto Global das Features:")
print("   Cada ponto = uma amostra do teste.")
print("   Cor = valor da feature (vermelho=alto, azul=baixo).")
print("   Posição X = impacto SHAP na predição.")
print()

fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(
    shap_values_classe1,
    X_test_sel,
    plot_type="dot",
    show=False
)
plt.title("SHAP Summary Plot — Classe 'Doença Cardíaca'", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Bar Plot - Importancia media absoluta
fig, ax = plt.subplots(figsize=(12, 6))
shap.summary_plot(
    shap_values_classe1,
    X_test_sel,
    plot_type="bar",
    show=False
)
plt.title("SHAP — Importância Média Absoluta das Features", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.show()

### 3.5.2 Explicação Local — Waterfall Plot

In [ ]:
# Explicacao local - Waterfall plot para uma predicao especifica
# Selecionar um paciente corretamente classificado como DOENTE
y_pred_sel = modelo_sel.predict(X_test_sel)
indices_doentes = np.where((y_test.values == 1) & (y_pred_sel == 1))[0]

if len(indices_doentes) > 0:
    idx_exemplo = indices_doentes[0]
else:
    idx_exemplo = 0

print(f"🔍 Explicação Local — Paciente #{idx_exemplo}:")
print(f"   Classe real:     {'Doente' if y_test.values[idx_exemplo] == 1 else 'Saudável'}")
print(f"   Classe predita:  {'Doente' if y_pred_sel[idx_exemplo] == 1 else 'Saudável'}")
print(f"   Probabilidade:   {modelo_sel.predict_proba(X_test_sel)[idx_exemplo]}")
print()
print("   Valores das features deste paciente:")
for feat_name, feat_val in X_test_sel.iloc[idx_exemplo].items():
    print(f"      {feat_name:<20} = {feat_val:.4f}")

In [ ]:
# Waterfall plot
print("📊 Waterfall Plot — Contribuição de cada feature para a predição:")
print("   Barras vermelhas: empurram a predição para 'Doente'")
print("   Barras azuis: empurram a predição para 'Saudável'")
print()

shap_explanation = shap.Explanation(
    values=shap_values_classe1[idx_exemplo],
    base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
    data=X_test_sel.iloc[idx_exemplo].values,
    feature_names=X_test_sel.columns.tolist()
)

fig, ax = plt.subplots(figsize=(12, 7))
shap.plots.waterfall(shap_explanation, show=False)
plt.title(f"Explicação Local — Paciente #{idx_exemplo}", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.show()

### 3.5.3 Explicação Local — Paciente Saudável (Contraexemplo)

In [ ]:
# Selecionar um paciente corretamente classificado como SAUDAVEL
indices_saudaveis = np.where((y_test.values == 0) & (y_pred_sel == 0))[0]

if len(indices_saudaveis) > 0:
    idx_saudavel = indices_saudaveis[0]
else:
    idx_saudavel = 1

print(f"🔍 Explicação Local — Paciente #{idx_saudavel} (Saudável):")
print(f"   Classe real:     {'Doente' if y_test.values[idx_saudavel] == 1 else 'Saudável'}")
print(f"   Classe predita:  {'Doente' if y_pred_sel[idx_saudavel] == 1 else 'Saudável'}")
print(f"   Probabilidade:   {modelo_sel.predict_proba(X_test_sel)[idx_saudavel]}")
print()

shap_explanation_s = shap.Explanation(
    values=shap_values_classe1[idx_saudavel],
    base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
    data=X_test_sel.iloc[idx_saudavel].values,
    feature_names=X_test_sel.columns.tolist()
)

fig, ax = plt.subplots(figsize=(12, 7))
shap.plots.waterfall(shap_explanation_s, show=False)
plt.title(f"Explicação Local — Paciente #{idx_saudavel} (Saudável)", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.show()

## 3.6 Comparação: Ranking SHAP vs. Importância RF vs. RFE

In [ ]:
# Comparacao dos rankings
shap_importancia = pd.Series(
    np.abs(shap_values_classe1).mean(axis=0),
    index=X_test_sel.columns
).sort_values(ascending=False)

rf_imp_sel = importancias[features_selecionadas].sort_values(ascending=False)

# Tabela comparativa
df_ranking = pd.DataFrame({
    "Rank SHAP": range(1, len(shap_importancia) + 1),
    "SHAP (mean |SHAP|)": shap_importancia.values,
}).set_index(shap_importancia.index)

df_ranking["Rank RF"] = [list(rf_imp_sel.index).index(f) + 1 if f in rf_imp_sel.index else "-"
                         for f in shap_importancia.index]
df_ranking["RF Importance"] = [rf_imp_sel.get(f, 0) for f in shap_importancia.index]

print("📊 Comparação de Rankings — SHAP vs. Random Forest:")
print("=" * 65)
df_ranking

In [ ]:
# Grafico comparativo dos rankings
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# SHAP importance
axes[0].barh(
    shap_importancia.index[::-1],
    shap_importancia.values[::-1],
    color="#9b59b6", edgecolor="white"
)
axes[0].set_xlabel("Mean |SHAP value|")
axes[0].set_title("Ranking SHAP", fontweight="bold", fontsize=13)

# RF importance (filtered)
axes[1].barh(
    rf_imp_sel.index[::-1],
    rf_imp_sel.values[::-1],
    color="#e67e22", edgecolor="white"
)
axes[1].set_xlabel("Gini Importance")
axes[1].set_title("Ranking Random Forest (Gini)", fontweight="bold", fontsize=13)

plt.suptitle("Comparação de Importância: SHAP vs. Random Forest",
             fontweight="bold", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## 📝 Resumo da Etapa 3 — Seleção de Features e Interpretabilidade

### Seleção de Atributos

| Item | Valor |
|------|-------|
| **Método Embedded** | Random Forest Feature Importance (Gini) |
| **Método Wrapper** | RFE (Recursive Feature Elimination) |
| **Features iniciais** | Todas as features após One-Hot Encoding |
| **Features selecionadas** | Subconjunto selecionado pelo RFE (critério: 85% da importância acumulada) |
| **Critério de corte** | Importância cumulativa ≥ 85% |

### Desempenho: Seleção de Features Melhorou?

A comparação entre os dois cenários (todas vs. selecionadas) revela:

- **Acurácia e F1-Score:** A seleção de features geralmente mantém desempenho competitivo com menos atributos, o que demonstra que as features removidas contribuíam pouco (ou adicionavam ruído).
- **Overfitting:** O gap entre acurácia de treino e teste tende a diminuir com a seleção, pois menos features = menos parâmetros = menor risco de memorização.
- **Eficiência:** Menos features significam treinamento mais rápido e modelos mais interpretáveis — crítico em contextos clínicos.

### SHAP vs. Features Selecionadas

A análise SHAP complementa a seleção de features ao:

1. **Validar a seleção:** As features com maiores valores SHAP coincidem em grande parte com as selecionadas pelo RFE e pela importância Gini do Random Forest, reforçando a robustez da seleção.
2. **Revelar direção do impacto:** Enquanto a importância Gini e o RFE indicam apenas *quanto* uma feature importa, o SHAP mostra *como* ela afeta a predição (positiva ou negativamente).
3. **Explicações locais:** Os waterfall plots mostram que o modelo não depende de uma única feature — a decisão é resultado de uma **combinação de fatores clínicos**, o que é medicamente coerente.

### Principais Features Identificadas

As features mais consistentemente importantes entre os três métodos são:
- `thalach` — Frequência cardíaca máxima
- `oldpeak` — Depressão do segmento ST
- `ca` — Número de vasos coloridos por fluoroscopia
- `cp_*` — Tipo de dor no peito
- `age` — Idade do paciente

Essas variáveis são **clinicamente relevantes** e amplamente reconhecidas na literatura médica como fatores de risco para doenças cardíacas.

### Próximos Passos

- **Etapa 4:** Treinamento de múltiplos modelos (Regressão Logística, SVM, Random Forest, Rede Neural MLP) com as features selecionadas.
- As features selecionadas serão utilizadas para treinamento nas etapas seguintes, otimizando a eficiência computacional e interpretabilidade.

---

# Etapa 4 — Classificação com Rede Neural MLP (TensorFlow/Keras)

Nesta etapa, construiremos uma **Rede Neural Artificial do tipo Multilayer Perceptron (MLP)** para classificar a presença de doença cardíaca. Cada hiperparâmetro será formalmente justificado, e o desempenho será rigorosamente comparado ao baseline de Random Forest da Etapa 3.

## 4.1 Justificativa Formal da Arquitetura MLP

Cada decisão arquitetural impacta a capacidade de aprendizado, convergência e generalização da rede. Abaixo, justificamos cada hiperparâmetro com fundamentação teórica.

---

### Camadas Ocultas: 3 camadas (64 → 32 → 16 neurônios)

**Justificativa:**
- Pelo **Teorema da Aproximação Universal** (Hornik, 1991), uma única camada oculta com neurônios suficientes pode aproximar qualquer função contínua. Contudo, redes mais profundas aprendem **representações hierárquicas** com menos neurônios totais.
- Com ~303 amostras e ~15-20 features (após One-Hot), 3 camadas oferecem capacidade suficiente sem risco excessivo de overfitting.
- A arquitetura em **funil decrescente** (64 → 32 → 16) força a rede a aprender representações progressivamente mais compactas e abstratas, atuando como um "encoder" implícito.
- **Regra prática:** O número de neurônios na primeira camada deve estar entre o número de features e o dobro. Com ~15 features selecionadas, 64 neurônios oferece margem para capturar interações não-lineares complexas.

### Função de Ativação: ReLU (camadas ocultas) + Sigmoid (saída)

**ReLU (Rectified Linear Unit):** $f(x) = \max(0, x)$
- Resolve o problema do **vanishing gradient** que afeta sigmoide/tanh em redes profundas: o gradiente de ReLU é 1 para $x > 0$, permitindo fluxo eficiente do gradiente.
- Computacionalmente eficiente (apenas uma comparação).
- Induz **esparsidade** natural (neurônios com entrada negativa têm saída 0), atuando como regularização implícita.

**Sigmoid (saída):** $\sigma(x) = \frac{1}{1 + e^{-x}}$
- Para classificação binária, a saída deve estar em $[0, 1]$ representando $P(y=1|x)$.
- Sigmoid mapeia qualquer valor real para esse intervalo, ideal para interpretar como probabilidade.

### Otimizador: Adam (Adaptive Moment Estimation)

Adam combina as vantagens de:
- **AdaGrad:** Learning rate adaptativo por parâmetro (bom para features esparsas como One-Hot).
- **RMSProp:** Média móvel exponencial dos gradientes ao quadrado.

A atualização dos pesos usa momentos de 1ª e 2ª ordem:

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$
$$\theta_{t+1} = \theta_t - \frac{\alpha}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

Isso permite convergência rápida e estável, mesmo com gradientes ruidosos.

### Learning Rate: 0.001

- Valor padrão recomendado pelo paper original de Adam (Kingma & Ba, 2015).
- Oferece bom equilíbrio entre velocidade de convergência e estabilidade.
- Complementado por `ReduceLROnPlateau` que reduz automaticamente quando a loss estagna.

### Loss Function: Binary Cross-Entropy

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N}[y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)]$$

- Função de perda padrão para classificação binária com saída sigmoide.
- Penaliza proporcionalmente à confiança em predições erradas (gradientes maiores para erros confiantes).

### Batch Size: 16

- Com ~242 amostras de treino (80% de 303), batches de 16 fornecem ~15 atualizações por época.
- Mini-batches pequenos introduzem ruído estocástico benefíco que ajuda a escapar de mínimos locais.
- Batches menores favorecem melhor generalização (Keskar et al., 2017).

### Épocas: 150 (com Early Stopping)

- 150 épocas como limite máximo, com **Early Stopping** (paciência=20) monitorando a loss de validação.
- O treinamento para automaticamente quando o modelo começa a sobreajustar, restaurando os melhores pesos.

### Regularização: Dropout (0.3) + BatchNormalization

- **Dropout (30%):** Desativa aleatoriamente 30% dos neurônios durante o treino, forçando a rede a aprender representações redundantes e reduzindo co-adaptação.
- **BatchNormalization:** Normaliza as ativações intermediarias, estabilizando e acelerando o treinamento (Ioffe & Szegedy, 2015).

## 4.2 Importações e Configuração

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.metrics import roc_curve, auc
import time

# Reprodutibilidade
tf.random.set_seed(RANDOM_STATE)

print(f"✅ TensorFlow {tf.__version__} carregado.")
print(f"   GPU disponível: {tf.config.list_physical_devices('GPU')}")

## 4.3 Preparação dos Dados para a MLP

In [ ]:
# Usar as features selecionadas na Etapa 3
print(f"📋 Dados para treinamento da MLP:")
print(f"   Features utilizadas: {len(features_selecionadas)} (selecionadas pelo RFE)")
print(f"   X_train: {X_train_sel.shape}")
print(f"   X_test:  {X_test_sel.shape}")
print()

# Converter para numpy (Keras espera arrays)
X_train_np = X_train_sel.values.astype("float32")
X_test_np = X_test_sel.values.astype("float32")
y_train_np = y_train.values.astype("float32")
y_test_np = y_test.values.astype("float32")

# Separar uma parcela do treino para validacao (15% do treino)
from sklearn.model_selection import train_test_split as tts
X_tr, X_val, y_tr, y_val = tts(
    X_train_np, y_train_np,
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=y_train_np
)

print(f"   Divisão interna (treino/validação):")
print(f"      X_treino:    {X_tr.shape}")
print(f"      X_validacao: {X_val.shape}")
print(f"      X_teste:     {X_test_np.shape}")

## 4.4 Construção da Arquitetura MLP

In [ ]:
def construir_mlp(input_dim, learning_rate=0.001):
    """
    Constroi uma MLP para classificacao binaria de doencas cardiacas.

    Arquitetura: Input -> [Dense+BN+ReLU+Dropout] x3 -> Sigmoid

    Parametros:
        input_dim (int): Numero de features de entrada.
        learning_rate (float): Taxa de aprendizado do Adam.

    Retorna:
        keras.Model: Modelo compilado.
    """
    modelo = keras.Sequential([
        # Camada de entrada
        layers.Input(shape=(input_dim,), name="input"),

        # Camada oculta 1: 64 neuronios
        layers.Dense(64, kernel_initializer="he_normal", name="dense_1"),
        layers.BatchNormalization(name="bn_1"),
        layers.Activation("relu", name="relu_1"),
        layers.Dropout(0.3, name="dropout_1"),

        # Camada oculta 2: 32 neuronios
        layers.Dense(32, kernel_initializer="he_normal", name="dense_2"),
        layers.BatchNormalization(name="bn_2"),
        layers.Activation("relu", name="relu_2"),
        layers.Dropout(0.3, name="dropout_2"),

        # Camada oculta 3: 16 neuronios
        layers.Dense(16, kernel_initializer="he_normal", name="dense_3"),
        layers.BatchNormalization(name="bn_3"),
        layers.Activation("relu", name="relu_3"),
        layers.Dropout(0.2, name="dropout_3"),

        # Camada de saida: 1 neuronio + sigmoid
        layers.Dense(1, activation="sigmoid", name="output")
    ], name="MLP_Heart_Disease")

    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="auc")
        ]
    )

    return modelo


# Construir o modelo
mlp = construir_mlp(input_dim=X_tr.shape[1], learning_rate=0.001)

# Sumario da arquitetura
mlp.summary()

In [ ]:
# Contagem de parametros
total_params = mlp.count_params()
trainable = sum(tf.keras.backend.count_params(w) for w in mlp.trainable_weights)
non_trainable = total_params - trainable

print(f"\n📊 Resumo da Arquitetura:")
print(f"   Parâmetros totais:     {total_params:,}")
print(f"   Parâmetros treináveis: {trainable:,}")
print(f"   Parâmetros fixos (BN): {non_trainable:,}")
print(f"   Razão amostras/parâmetros: {X_tr.shape[0] / trainable:.2f}")
print()
if X_tr.shape[0] / trainable < 1:
    print("   ⚠️ Razão < 1: risco de overfitting. Dropout e Early Stopping são essenciais.")
else:
    print("   ✅ Razão adequada para treinamento.")

## 4.5 Configuração de Callbacks e Treinamento

In [ ]:
# Hiperparametros
EPOCHS = 150
BATCH_SIZE = 16
PATIENCE_ES = 20    # Early Stopping
PATIENCE_LR = 10    # Reduce LR on Plateau

# Callbacks
callbacks_list = [
    # Early Stopping: para quando val_loss nao melhora
    callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE_ES,
        restore_best_weights=True,
        verbose=1
    ),
    # Reduce LR: reduz learning rate quando val_loss estagna
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=PATIENCE_LR,
        min_lr=1e-6,
        verbose=1
    )
]

print("🔧 Hiperparâmetros de Treinamento:")
print(f"   Épocas máximas:    {EPOCHS}")
print(f"   Batch size:         {BATCH_SIZE}")
print(f"   Early Stopping:     paciência = {PATIENCE_ES} épocas")
print(f"   ReduceLR:           fator 0.5 a cada {PATIENCE_LR} épocas sem melhora")
print(f"   Learning rate:      0.001")
print(f"   Otimizador:         Adam")
print(f"   Loss:               Binary Cross-Entropy")

In [ ]:
# Treinamento
print("🚀 Iniciando treinamento da MLP...")
print("=" * 50)

inicio = time.time()

historico = mlp.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_list,
    verbose=1
)

tempo_mlp = time.time() - inicio

epocas_executadas = len(historico.history["loss"])
print(f"\n✅ Treinamento concluído!")
print(f"   Épocas executadas: {epocas_executadas}/{EPOCHS}")
print(f"   Tempo total: {tempo_mlp:.2f}s")
print(f"   Melhor val_loss: {min(historico.history['val_loss']):.4f}")

## 4.6 Curvas de Aprendizado

In [ ]:
# Extrair metricas do historico
hist = historico.history
epocas_range = range(1, epocas_executadas + 1)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Curvas de Aprendizado da MLP", fontsize=16, fontweight="bold", y=1.01)

# --- Loss ---
axes[0, 0].plot(epocas_range, hist["loss"], label="Treino", color="#3498db", linewidth=2)
axes[0, 0].plot(epocas_range, hist["val_loss"], label="Validação", color="#e74c3c", linewidth=2)
axes[0, 0].set_title("Binary Cross-Entropy Loss", fontweight="bold")
axes[0, 0].set_xlabel("Época")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(True, alpha=0.3)
best_epoch = np.argmin(hist["val_loss"]) + 1
axes[0, 0].axvline(x=best_epoch, color="green", linestyle="--", alpha=0.5, label=f"Melhor época ({best_epoch})")
axes[0, 0].legend(fontsize=10)

# --- Accuracy ---
axes[0, 1].plot(epocas_range, hist["accuracy"], label="Treino", color="#3498db", linewidth=2)
axes[0, 1].plot(epocas_range, hist["val_accuracy"], label="Validação", color="#e74c3c", linewidth=2)
axes[0, 1].set_title("Accuracy", fontweight="bold")
axes[0, 1].set_xlabel("Época")
axes[0, 1].set_ylabel("Accuracy")
axes[0, 1].legend(fontsize=11)
axes[0, 1].grid(True, alpha=0.3)

# --- Precision ---
axes[1, 0].plot(epocas_range, hist["precision"], label="Treino", color="#3498db", linewidth=2)
axes[1, 0].plot(epocas_range, hist["val_precision"], label="Validação", color="#e74c3c", linewidth=2)
axes[1, 0].set_title("Precision", fontweight="bold")
axes[1, 0].set_xlabel("Época")
axes[1, 0].set_ylabel("Precision")
axes[1, 0].legend(fontsize=11)
axes[1, 0].grid(True, alpha=0.3)

# --- AUC ---
axes[1, 1].plot(epocas_range, hist["auc"], label="Treino", color="#3498db", linewidth=2)
axes[1, 1].plot(epocas_range, hist["val_auc"], label="Validação", color="#e74c3c", linewidth=2)
axes[1, 1].set_title("ROC-AUC", fontweight="bold")
axes[1, 1].set_xlabel("Época")
axes[1, 1].set_ylabel("AUC")
axes[1, 1].legend(fontsize=11)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.6.1 Evolução do F1-Score por Época

In [ ]:
# Calcular F1-Score por epoca a partir de precision e recall do historico
f1_treino = []
f1_valid = []

for p, r in zip(hist["precision"], hist["recall"]):
    f1 = 2 * (p * r) / (p + r + 1e-8)
    f1_treino.append(f1)

for p, r in zip(hist["val_precision"], hist["val_recall"]):
    f1 = 2 * (p * r) / (p + r + 1e-8)
    f1_valid.append(f1)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(epocas_range, f1_treino, label="F1-Score Treino", color="#3498db", linewidth=2)
ax.plot(epocas_range, f1_valid, label="F1-Score Validação", color="#e74c3c", linewidth=2)
ax.axvline(x=best_epoch, color="green", linestyle="--", alpha=0.6, label=f"Melhor época ({best_epoch})")
ax.set_title("Evolução do F1-Score por Época", fontweight="bold", fontsize=14)
ax.set_xlabel("Época")
ax.set_ylabel("F1-Score")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

print(f"   Melhor F1-Score (validação): {max(f1_valid):.4f} (na época {np.argmax(f1_valid) + 1})")

---

## 4.7 Avaliação da MLP no Conjunto de Teste

In [ ]:
# Predicoes no conjunto de teste
y_pred_proba_mlp = mlp.predict(X_test_np, verbose=0).flatten()
y_pred_mlp = (y_pred_proba_mlp >= 0.5).astype(int)

# Metricas
metricas_mlp = {
    "Cenario": "MLP (Keras)",
    "N. Features": X_test_sel.shape[1],
    "Accuracy": accuracy_score(y_test_np, y_pred_mlp),
    "Precision": precision_score(y_test_np, y_pred_mlp),
    "Recall": recall_score(y_test_np, y_pred_mlp),
    "F1-Score": f1_score(y_test_np, y_pred_mlp),
    "ROC-AUC": roc_auc_score(y_test_np, y_pred_proba_mlp)
}

print("📊 Métricas da MLP no Conjunto de Teste:")
print("=" * 50)
for metrica, valor in metricas_mlp.items():
    if metrica not in ["Cenario", "N. Features"]:
        print(f"   {metrica:<12} {valor:.4f}")
print()
print("\n📋 Classification Report:")
print(classification_report(
    y_test_np, y_pred_mlp,
    target_names=["Saudável (0)", "Doente (1)"]
))

### 4.7.1 Matriz de Confusão da MLP

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusao
cm_mlp = confusion_matrix(y_test_np, y_pred_mlp)
disp_mlp = ConfusionMatrixDisplay(cm_mlp, display_labels=["Saudável", "Doente"])
disp_mlp.plot(ax=axes[0], cmap="Purples", values_format="d")
axes[0].set_title("Matriz de Confusão — MLP", fontweight="bold", fontsize=13)

# Curva ROC
fpr, tpr, _ = roc_curve(y_test_np, y_pred_proba_mlp)
roc_auc_val = auc(fpr, tpr)

axes[1].plot(fpr, tpr, color="#9b59b6", linewidth=2.5,
             label=f"MLP (AUC = {roc_auc_val:.3f})")

# Adicionar curva do RF para comparacao
y_proba_rf = modelo_sel.predict_proba(X_test_sel)[:, 1]
fpr_rf, tpr_rf, _ = roc_curve(y_test_np, y_proba_rf)
roc_auc_rf = auc(fpr_rf, tpr_rf)
axes[1].plot(fpr_rf, tpr_rf, color="#2ecc71", linewidth=2.5,
             label=f"RF Baseline (AUC = {roc_auc_rf:.3f})")

axes[1].plot([0, 1], [0, 1], "k--", alpha=0.3, label="Aleatório (AUC = 0.5)")
axes[1].set_xlabel("Taxa de Falso Positivo (FPR)")
axes[1].set_ylabel("Taxa de Verdadeiro Positivo (TPR)")
axes[1].set_title("Curva ROC — MLP vs. Random Forest", fontweight="bold", fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 4.8 Comparação: MLP vs. Random Forest Baseline

In [ ]:
# Medir tempo do Random Forest para comparacao
inicio_rf = time.time()
rf_comparacao = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_comparacao.fit(X_train_sel.values, y_train.values)
tempo_rf = time.time() - inicio_rf

y_pred_rf_comp = rf_comparacao.predict(X_test_sel.values)
y_proba_rf_comp = rf_comparacao.predict_proba(X_test_sel.values)[:, 1]

metricas_rf_comp = {
    "Cenario": "Random Forest (Baseline)",
    "N. Features": X_test_sel.shape[1],
    "Accuracy": accuracy_score(y_test.values, y_pred_rf_comp),
    "Precision": precision_score(y_test.values, y_pred_rf_comp),
    "Recall": recall_score(y_test.values, y_pred_rf_comp),
    "F1-Score": f1_score(y_test.values, y_pred_rf_comp),
    "ROC-AUC": roc_auc_score(y_test.values, y_proba_rf_comp),
    "Tempo (s)": tempo_rf
}

metricas_mlp_comp = metricas_mlp.copy()
metricas_mlp_comp["Tempo (s)"] = tempo_mlp

# Tabela comparativa
df_final = pd.DataFrame([metricas_rf_comp, metricas_mlp_comp]).set_index("Cenario")

print("📊 Comparação Final: MLP vs. Random Forest:")
print("=" * 70)
df_final

In [ ]:
# Grafico comparativo final
metricas_comparar = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# --- Metricas ---
x = np.arange(len(metricas_comparar))
largura = 0.35

bars_rf = axes[0].bar(
    x - largura/2,
    [metricas_rf_comp[m] for m in metricas_comparar],
    largura, label="Random Forest",
    color="#2ecc71", edgecolor="white", linewidth=1.2
)
bars_mlp = axes[0].bar(
    x + largura/2,
    [metricas_mlp_comp[m] for m in metricas_comparar],
    largura, label="MLP (Keras)",
    color="#9b59b6", edgecolor="white", linewidth=1.2
)

for bars in [bars_rf, bars_mlp]:
    for bar in bars:
        h = bar.get_height()
        axes[0].text(
            bar.get_x() + bar.get_width() / 2, h + 0.005,
            f"{h:.3f}", ha="center", va="bottom",
            fontweight="bold", fontsize=9
        )

axes[0].set_xticks(x)
axes[0].set_xticklabels(metricas_comparar, fontsize=11)
axes[0].set_ylim(0, 1.15)
axes[0].set_ylabel("Score")
axes[0].set_title("Métricas de Classificação", fontweight="bold", fontsize=13)
axes[0].legend(fontsize=11)
axes[0].axhline(y=1.0, color="gray", linestyle="--", alpha=0.3)

# --- Tempo de treinamento ---
modelos = ["Random Forest", "MLP (Keras)"]
tempos = [tempo_rf, tempo_mlp]
cores_tempo = ["#2ecc71", "#9b59b6"]

bars_t = axes[1].bar(modelos, tempos, color=cores_tempo, edgecolor="white", linewidth=1.5)
for bar, t in zip(bars_t, tempos):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
        f"{t:.2f}s", ha="center", va="bottom", fontweight="bold", fontsize=13
    )
axes[1].set_ylabel("Tempo (segundos)")
axes[1].set_title("Custo Computacional (Treinamento)", fontweight="bold", fontsize=13)

plt.suptitle("Comparação Final: MLP vs. Random Forest",
             fontweight="bold", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Matrizes de confusao lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RF
cm_rf = confusion_matrix(y_test.values, y_pred_rf_comp)
disp_rf = ConfusionMatrixDisplay(cm_rf, display_labels=["Saudável", "Doente"])
disp_rf.plot(ax=axes[0], cmap="Greens", values_format="d")
axes[0].set_title(f"Random Forest (F1={metricas_rf_comp['F1-Score']:.3f})",
                  fontweight="bold", fontsize=12)

# MLP
cm_mlp_c = confusion_matrix(y_test_np, y_pred_mlp)
disp_mlp_c = ConfusionMatrixDisplay(cm_mlp_c, display_labels=["Saudável", "Doente"])
disp_mlp_c.plot(ax=axes[1], cmap="Purples", values_format="d")
axes[1].set_title(f"MLP Keras (F1={metricas_mlp_comp['F1-Score']:.3f})",
                  fontweight="bold", fontsize=12)

plt.suptitle("Matrizes de Confusão — RF vs. MLP", fontweight="bold", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## 📝 Resumo da Etapa 4 — Classificação com MLP

### Arquitetura Implementada

| Componente | Configuração | Justificativa |
|-----------|-------------|---------------|
| **Camadas ocultas** | 3 (64 → 32 → 16) | Funil decrescente para abstração hierárquica |
| **Ativação oculta** | ReLU | Evita vanishing gradient; induz esparsidade |
| **Ativação saída** | Sigmoid | Classificação binária: P(doente) em [0,1] |
| **Regularização** | Dropout (0.3) + BatchNorm | Previne overfitting + estabiliza treinamento |
| **Otimizador** | Adam (lr=0.001) | Momentos adaptativos; convergência rápida |
| **Loss** | Binary Cross-Entropy | Padrão para classificação binária |
| **Batch size** | 16 | Ruído estocástico benéfico; melhor generalização |
| **Épocas** | 150 (Early Stopping, paciência=20) | Para automaticamente ao detectar overfitting |

### MLP vs. Random Forest — Discussão

#### Desempenho
- Ambos os modelos apresentam desempenho competitivo para este dataset de tamanho reduzido (~303 amostras).
- O Random Forest tende a ser mais robusto out-of-the-box em datasets tabulares pequenos, pois não requer ajuste fino de hiperparâmetros como a MLP.
- A MLP pode potencialmente superar o RF em datasets maiores, onde a capacidade de aprender interações não-lineares complexas é mais valorizada.

#### Custo Computacional
- O **Random Forest** treina significativamente mais rápido (paralelismo nativo de árvores independentes).
- A **MLP** requer mais tempo devido à otimização iterativa via backpropagation ao longo de múltiplas épocas.
- Em contextos clínicos com necessidade de retraining frequente, o RF pode ser mais prático.

#### Overfitting
- A MLP é mais suscetível a overfitting em datasets pequenos (alta razão parâmetros/amostras).
- As técnicas de regularização (Dropout, BatchNorm, Early Stopping) mitigam esse risco.
- As curvas de aprendizado mostram se houve divergência entre treino e validação.

#### Interpretabilidade
- O RF oferece importância de features nativa (Gini Importance).
- A MLP é uma "caixa preta" por natureza, mas SHAP (Etapa 3) pode ser aplicado a ela também.

### Conclusão

Para este dataset específico (pequeno, tabular, com features clínicas), o Random Forest oferece a melhor relação custo-benefício. A MLP demonstra o potencial de redes neurais e serve como base para experimentação com arquiteturas mais avançadas (ex: redes com atenção, AutoML).

### Próximos Passos

- **Etapa 5 (opcional):** Otimização de hiperparâmetros (GridSearch/Optuna) e validação cruzada estratégica.
- **Deploy:** Exportar o melhor modelo para produção (ONNX, TensorFlow Lite, ou pickle para RF).